# Tutorial: Halo Photometry of Aldebaran (*K2* Campaign 13)

This notebook demonstrates how to extract halo photometry of the bright red giant Aldebaran (*K2* Campaign 13; EPIC 200173843) with `halophot` v2.0+ and `lightkurve`, inspect the pixel weightmap, and save/reload a lossless `HALOCVER = 1` FITS product.

### `halophot` v2.0+ Functional `lightkurve` API

In `halophot` v2.0+, the recommended entry point `halophot.extract_halo_lightcurve` operates directly on any standard `lightkurve.TargetPixelFile` (without custom subclassing) and returns a standard `KeplerLightCurve` or `TessLightCurve` with:

- coherent `flux_err` propagating pixel, projected `SMEAR_ERR`, and `BKG_CHOL` / `BKG_BASIS` uncertainties,
- positive-flux weightmap-weighted centroids (`centroid_col`, `centroid_row`) alongside preserved `pos_corr1` / `pos_corr2` columns,
- non-transposed `(ny, nx)` (or `(n_splits, ny, nx)`) simplex weights in `lc.meta['WEIGHTMAP']`.

Always serialize halo light curves via `halophot.write_fits(lc, path)` rather than `lc.to_fits(path)` so `HALOCVER = 1` extensions (`WEIGHTMAP`, `WEIGHTMAP_ERR`, `SPLIT_TIMES`) and `POS_CORR1`/`POS_CORR2` are preserved.

In [ ]:
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable

import halophot
import lightkurve as lk

mpl.rcParams["image.origin"] = "lower"
mpl.rcParams["figure.figsize"] = (10.0, 10.0)
mpl.rcParams["font.size"] = 14
mpl.rcParams["savefig.dpi"] = 150

print("lightkurve:", lk.__version__)
print("halophot:", halophot.__version__)

## 1. Download and inspect the *K2* Target Pixel File

Download Aldebaran's *K2* Campaign 13 Target Pixel File via `lightkurve` (or pass a local FITS path to `lk.read(...)`).

In [ ]:
tpf = lk.search_targetpixelfile("EPIC 200173843", mission="K2", campaign=13).download()
tpf.plot(scale="log")

## 2. Extract split-segment halo photometry

Passing `split_times=[3010.0]` optimizes independent simplex weights in two segments across the campaign, returning a stitched `KeplerLightCurve` with stacked `(2, ny, nx)` weights in `corr_lc.meta['WEIGHTMAP']` and block-jackknife uncertainties in `corr_lc.meta['WEIGHTMAP_ERR']`.

In [ ]:
corr_lc = halophot.extract_halo_lightcurve(
    tpf,
    split_times=[3010.0],
    thresh=0.5,
    sigclip=True,
    compute_uncertainty=True,
)
weightmap = corr_lc.meta["WEIGHTMAP"][0]
print("Segments:", corr_lc.meta["WEIGHTMAP"].shape, "Cadences:", len(corr_lc))

## 3. Plot the extracted light curve and pixel weightmap

In `halophot` v2.0+, `WEIGHTMAP` uses standard `(ny, nx)` image orientation (no transpose required).

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

ax1.errorbar(
    corr_lc.time.value,
    corr_lc.flux.value,
    yerr=corr_lc.flux_err.value,
    fmt=".",
    ms=3,
    alpha=0.6,
    color="k",
    ecolor="0.7",
)
ax1.set_xlabel("Time (BKJD, days)")
ax1.set_ylabel("Normalized Flux")
ax1.set_title("Aldebaran (K2 C13) Halo Light Curve")

im = ax2.imshow(np.log10(np.where(weightmap > 0, weightmap, np.nan)), origin="lower", cmap="viridis")
divider = make_axes_locatable(ax2)
cax = divider.append_axes("right", size="5%", pad=0.05)
fig.colorbar(im, cax=cax, label=r"$\log_{10}(\mathrm{weight})$")
ax2.set_xlabel("Column (px)")
ax2.set_ylabel("Row (px)")
ax2.set_title("TV-min Weightmap (Segment 0)")
plt.tight_layout()

## 4. Save and reload as a `HALOCVER = 1` FITS product

Use `halophot.write_fits` so `POS_CORR1`, `POS_CORR2`, `WEIGHTMAP`, `WEIGHTMAP_ERR`, and `SPLIT_TIMES` round-trip losslessly for downstream systematic detrending (e.g. `lightkurve` SFF / Savitzky–Golay detrending or `k2sc` on the saved FITS file).

In [ ]:
out_fits = "aldebaran_c13_halo_lc.fits"
halophot.write_fits(corr_lc, out_fits, overwrite=True)

lc_reloaded = lk.read(out_fits)
wmap_reloaded = halophot.read_weightmap(lc_reloaded)
splits_reloaded = halophot.read_split_times(lc_reloaded)
print("Reloaded columns:", lc_reloaded.colnames)
print("Reloaded WEIGHTMAP shape:", wmap_reloaded.shape, "SPLIT_TIMES:", splits_reloaded)